# 📊 Forest Inspection - Evaluation Notebook

Đánh giá model: metrics, per-sequence analysis, deforestation analysis.

In [ ]:
import sys
import torch
import numpy as np
from torch.utils.data import DataLoader
from pathlib import Path

sys.path.insert(0, '.')

from src.data.dataset import ForestDataset, NUM_CLASSES, CLASS_NAMES
from src.data.transforms import get_val_transforms
from src.data.splits import get_split, SEQUENCE_INFO
from src.models import build_model
from src.evaluation.evaluator import Evaluator
from src.evaluation.visualize import visualize_prediction
from src.evaluation.deforestation import (
    compute_deforestation_index,
    analyze_sequence_deforestation,
    plot_deforestation_comparison,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Load Model

In [ ]:
DATA_ROOT = 'data/forest_sunny'
CHECKPOINT_PATH = 'outputs/checkpoints/best_model.pth'  # Update this
MODEL_NAME = 'unet'
ENCODER = 'resnet34'

# Build model and load weights
model = build_model(MODEL_NAME, encoder=ENCODER, num_classes=NUM_CLASSES)

if Path(CHECKPOINT_PATH).exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded checkpoint: {CHECKPOINT_PATH}")
    print(f"  Epoch: {ckpt.get('epoch', '?')}, mIoU: {ckpt.get('metric', '?'):.4f}")
else:
    print(f'⚠️ Checkpoint not found: {CHECKPOINT_PATH}')
    print('  Using randomly initialized model (for testing pipeline only)')

## 2. Test Set Evaluation

In [ ]:
# Load test data
split = get_split('cross_sequence')
val_tf = get_val_transforms(img_size=(512, 512))

test_ds = ForestDataset(DATA_ROOT, split['test'], transform=val_tf)
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=2)

print(f'Test set: {len(test_ds)} samples from {split["test"]}')

# Evaluate
evaluator = Evaluator(model, test_loader, device=device)
results = evaluator.evaluate()
evaluator.print_results()

## 3. Per-Sequence Analysis (Altitude & Pitch Impact)

In [ ]:
import matplotlib.pyplot as plt

# Evaluate on each sequence
all_seqs = [f'seq{i}' for i in range(1, 10)]
per_seq = evaluator.evaluate_per_sequence(DATA_ROOT, all_seqs, val_tf, batch_size=8)

# Build mIoU matrix: altitude × pitch
altitudes = [30, 50, 80]
pitches = [0, -60, -90]
miou_matrix = np.zeros((3, 3))

for seq_name, metrics in per_seq.items():
    info = SEQUENCE_INFO[seq_name]
    row = altitudes.index(info['altitude'])
    col = pitches.index(info['pitch'])
    miou_matrix[row, col] = metrics['miou']

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(miou_matrix, cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(3))
ax.set_xticklabels([f'{p}°' for p in pitches])
ax.set_yticks(range(3))
ax.set_yticklabels([f'{a}m' for a in altitudes])
ax.set_xlabel('Pitch Angle')
ax.set_ylabel('Altitude')
ax.set_title('mIoU by Altitude × Pitch', fontsize=14)

for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{miou_matrix[i,j]:.3f}', ha='center', va='center', fontsize=14, fontweight='bold')

plt.colorbar(im, label='mIoU')
plt.tight_layout()
plt.show()

## 4. Prediction Visualization

In [ ]:
# Save prediction visualizations
evaluator.save_predictions(output_dir='outputs/predictions', num_samples=10)

## 5. Deforestation Analysis

In [ ]:
# Analyze deforestation per sequence
deforest_results = {}
for seq in ['seq3', 'seq6', 'seq9']:  # Top-down views (best for analysis)
    ds = ForestDataset(DATA_ROOT, [seq], transform=val_tf)
    loader = DataLoader(ds, batch_size=8, shuffle=False)
    deforest_results[seq] = analyze_sequence_deforestation(model, loader, device, seq)

# Plot comparison
plot_deforestation_comparison(deforest_results, save_path='outputs/deforestation_analysis.png')

## 6. Confusion Matrix

In [ ]:
import seaborn as sns

cm = evaluator.metrics.get_confusion_matrix()
# Normalize by row (true class)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Normalized Confusion Matrix', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()